## trying different optimising techniques
Roberta_finetuning is the final model


In [ ]:
#this is not the Main roberta code this is the initial setup used to evaluate different Hyperparameters and optimisation techniques




# ══════════════════════════════════════════════════════════════════════════
#  Mount Google Drive + upload dataset
# ══════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount("/content/drive")
DATA_PATH = "/content/drive/MyDrive/specality_classification/mtsamples.csv"


# ══════════════════════════════════════════════════════════════════════════
# Imports
# ══════════════════════════════════════════════════════════════════════════
import re, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import Dataset, DataLoader,WeightedRandomSampler
from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup,
)
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running on: cpu


In [7]:
# ══════════════════════════════════════════════════════════════════════════
# Load & preprocess
# ══════════════════════════════════════════════════════════════════════════
df = pd.read_csv(DATA_PATH)
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

df["medical_specialty"] = df["medical_specialty"].str.strip()

counts = df["medical_specialty"].value_counts()
df["label"] = df["medical_specialty"].apply(
    lambda x: x if counts[x] >= 20 else "Other"
)

df["text"] = (
    df["transcription"].fillna("") + " " + df["description"].fillna("")
).str.strip()

df = df[df["text"].str.len() > 30].reset_index(drop=True)

print(f"Rows   : {len(df)}")
print(f"Classes: {df['label'].nunique()}")
print(df["label"].value_counts().to_string())

# ══════════════════════════════════════════════════════════════════════════
# Label encode + stratified split  (70 / 15 / 15)
# ══════════════════════════════════════════════════════════════════════════
le = LabelEncoder()
y  = le.fit_transform(df["label"].values)
X  = df["text"].values
NUM_CLASSES = len(le.classes_)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)
print(f"Train:{len(X_train)}  Val:{len(X_val)}  Test:{len(X_test)}")



# ══════════════════════════════════════════════════════════════════════════
# Tokeniser + Dataset
# ══════════════════════════════════════════════════════════════════════════
MODEL_NAME = "roberta-base"
MAX_LEN    = 512
BATCH_SIZE = 16

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

class MedDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(
            list(texts),
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids"      : self.enc["input_ids"][idx],
            "attention_mask" : self.enc["attention_mask"][idx],
            "labels"         : self.labels[idx],
        }

train_ds = MedDataset(X_train, y_train)
val_ds   = MedDataset(X_val,   y_val)
test_ds  = MedDataset(X_test,  y_test)

# Sampler removed — sqrt weighted loss handles imbalance instead.
# Using both sampler + weighted loss causes train/val distribution mismatch
# which leads to overfitting (train F1 >> val F1).
label_counts = np.bincount(y_train)
raw_weights  = len(y_train) / (NUM_CLASSES * label_counts)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True)
print("Dataloaders ready ")


Rows   : 4987
Classes: 30
label
Surgery                          1098
Consult - History and Phy.        516
Cardiovascular / Pulmonary        371
Orthopedic                        355
Radiology                         273
General Medicine                  259
Gastroenterology                  228
Neurology                         223
SOAP / Chart / Progress Notes     166
Obstetrics / Gynecology           160
Urology                           157
Other                             125
Discharge Summary                 108
ENT - Otolaryngology               96
Neurosurgery                       94
Hematology - Oncology              90
Ophthalmology                      83
Nephrology                         81
Emergency Room Reports             75
Pediatrics - Neonatal              70
Pain Management                    62
Psychiatry / Psychology            53
Office Notes                       50
Podiatry                           47
Dermatology                        29
Dentistry         

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Dataloaders ready 


In [10]:

# ══════════════════════════════════════════════════════════════════════════
# Model + optimiser
# ══════════════════════════════════════════════════════════════════════════
EPOCHS   = 10    # more epochs — model was still converging at epoch 5
LR       = 2e-5
WARMUP   = 0.1

model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_CLASSES,
    hidden_dropout_prob=0.2,              # increased from 0.1 — reduces overfitting
    attention_probs_dropout_prob=0.2,     # increased from 0.1
)
model.to(device)

total_steps  = len(train_dl) * EPOCHS
warmup_steps = int(total_steps * WARMUP)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# Sqrt-softened class weights for loss:
# Reduces the 55x Surgery/SleepMedicine ratio to ~7x so rare-class
# gradients don't overwhelm the optimiser and cause collapse.
# No class weights in the loss — let the model learn naturally.
# Weighted loss + small dataset was slowing convergence to a crawl.
# The stratified split already ensures all classes appear in training.
loss_fn = torch.nn.CrossEntropyLoss()

print(f"Total steps : {total_steps}  |  Warmup: {warmup_steps}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total steps : 2190  |  Warmup: 219


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Training loop
# ══════════════════════════════════════════════════════════════════════════
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            ids   = batch["input_ids"].to(device)
            mask  = batch["attention_mask"].to(device)
            labs  = batch["labels"].to(device)
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss   = loss_fn(logits, labs)
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            total_loss += loss.item()
            all_preds.extend(logits.argmax(-1).cpu().numpy())
            all_labels.extend(labs.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    mf1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / len(loader), acc, mf1

best_val_f1 = 0.0
history     = []
PATIENCE    = 2       # stop if val F1 doesn't improve for 2 epochs in a row
no_improve  = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_f1 = run_epoch(train_dl, train=True)
    vl_loss, vl_acc, vl_f1 = run_epoch(val_dl,   train=False)
    history.append(dict(epoch=epoch,
                        tr_loss=tr_loss, tr_acc=tr_acc, tr_f1=tr_f1,
                        vl_loss=vl_loss, vl_acc=vl_acc, vl_f1=vl_f1))
    print(f"Epoch {epoch}/{EPOCHS} | "
          f"Train loss={tr_loss:.4f} acc={tr_acc:.4f} F1={tr_f1:.4f} | "
          f"Val   loss={vl_loss:.4f} acc={vl_acc:.4f} F1={vl_f1:.4f}")
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        no_improve  = 0
        torch.save(model.state_dict(), "/content/roberta_best.pt")
        print(f"  ✓ Saved best checkpoint (val F1={best_val_f1:.4f})")
    else:
        no_improve += 1
        print(f"  No improvement ({no_improve}/{PATIENCE})")
        if no_improve >= PATIENCE:
            print("  Early stopping triggered.")
            break

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Test evaluation
# ══════════════════════════════════════════════════════════════════════════
# Re-define le in case kernel lost it between cells
le = LabelEncoder()
le.fit(df["label"].values)
NUM_CLASSES = len(le.classes_)

model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_CLASSES,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2,
)
model.to(device)
model.load_state_dict(torch.load("/content/roberta_best.pt", map_location=device))
_, test_acc, test_f1 = run_epoch(test_dl, train=False)
print(f"\n{'='*55}")
print(f"TEST Accuracy : {test_acc:.4f}")
print(f"TEST Macro-F1 : {test_f1:.4f}")
print(f"{'='*55}")

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_dl:
        logits = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device)
        ).logits
        all_preds.extend(logits.argmax(-1).cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print(classification_report(all_labels, all_preds,
                             target_names=le.classes_, zero_division=0))